# DCJ quarterly sales workbooks

This notebook inspects issues 128–156, covering March 2019 through March 2026. The NSW Department of Communities and Justice (DCJ) published all 29 quarters. Its Greater Sydney grouping contains 34 LGAs across three rings. Wollondilly has no published strata median in 2019. The June issue omits the row, and the other three issues report a dash.

Start Jupyter from the repository root or run the nbconvert command in T01. The notebook reads the original workbooks under `data/raw/dcj_sales/`. Quarters use the first day of the quarter when shown as a date. [DCJ sales explanatory notes](https://dcj.nsw.gov.au/about-us/families-and-communities-statistics/housing-rent-and-sales/explanatory-notes.html)

## 1. Files

List the workbooks, their byte sizes and sheet names. Read the values from the `LGA` sheet and keep their Excel row positions so each observation can be traced to the source.

In [1]:
from pathlib import Path
import re
from string import ascii_uppercase

import pandas as pd
from IPython.display import display
from openpyxl import load_workbook

raw_dir = Path("data/raw")
if Path.cwd().name == "notebooks":
    raw_dir = Path("../data/raw")
sales_dir = raw_dir / "dcj_sales"

files = []
lga_sheets = {}
for path in sorted(sales_dir.glob("*.xlsx")):
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        files.append({
            "filename": path.name,
            "bytes": path.stat().st_size,
            "sheets": ", ".join(workbook.sheetnames),
        })
        lga_sheets[path.name] = list(workbook["LGA"].values)
    finally:
        workbook.close()

file_inventory = pd.DataFrame(files).set_index("filename")
display(file_inventory)

,bytes,sheets
filename,,
Sales_tables_September_2024_quarter.xlsx,471111,"Explanatory Notes, LGA, Postcode"
issue-128-sales-tables-march-2019.xlsx,380498,"Explanatory Notes, LGA, Postcode"
issue-129-sales-tables-june-2019.xlsx,550885,"Explanatory Notes, LGA, Postcode"
issue-130-sales-tables-september-2019.xlsx,389253,"Explanatory Notes, LGA, Postcode"
issue-131-sales-tables-december-2019.xlsx,393825,"Explanatory Notes, LGA, Postcode"
issue-132-sales-tables-march-2020.xlsx,631019,"Explanatory Notes, LGA, Postcode"
issue-133-sales-tables-june-2020.xlsx,595512,"Explanatory Notes, LGA, Postcode"
issue-134-sales-tables-september-2020.xlsx,604710,"Explanatory Notes, LGA, Postcode"
issue-135-sales-tables-december-2020.xlsx,610082,"Explanatory Notes, LGA, Postcode"


## 2. Layout

Inspect the latest workbook’s opening rows and the exact Excel headers. Its sales table has one reporting quarter. Dwelling type is recorded in column E as `Non Strata`, `Strata` or `Total`. The layout is long by dwelling type and wide by measure, with separate columns for the price statistics, sales count and quarterly and annual change. Prices are in thousands of dollars.

Find the header in every issue because its row number changes. Read the reporting quarter from the title in older files and the period line in newer files.

In [2]:
representative = "sales-tables-march-2026-quarter.xlsx"
columns = [
    "gmr", "greater_sydney", "ring", "lga", "dwelling_type",
    "first_quartile_price", "median_price", "third_quartile_price",
    "mean_price", "sales_count", "quarterly_median_change", "annual_median_change",
    "quarterly_count_change", "annual_count_change",
]
excel_columns = list(ascii_uppercase[:len(columns)])

layout = []
for filename, rows in lga_sheets.items():
    header_index = next(index for index, row in enumerate(rows) if row[4] == "DwellingType")
    period_caption = rows[0][0]
    if rows[1][0].startswith("Reporting period:"):
        period_caption = rows[1][0]
    # Both caption styles end in the quarter-end month and year.
    end_month, year = re.search(
        r"(March|June|September|December)(?: Quarter)? (\d{4})$", period_caption
    ).groups()
    quarter_end = pd.to_datetime(f"{end_month} {year}", format="%B %Y")
    layout.append({
        "filename": filename,
        "quarter": quarter_end.to_period("Q"),
        "quarter_date": quarter_end.to_period("Q").start_time,
        "header_row": header_index + 1,
        "period_caption": period_caption,
    })

layout = pd.DataFrame(layout).sort_values("quarter").set_index("filename")
latest_rows = lga_sheets[representative]
header_row = layout.loc[representative, "header_row"]
for excel_row, row in enumerate(latest_rows[:header_row - 1], start=1):
    if row[0] is not None:
        print(f"A{excel_row}  {row[0]}")

print(f"Header row in {representative} is {header_row}.")
display(pd.DataFrame({
    "excel_column": excel_columns,
    "source_header": [label.replace("\n", " ") for label in latest_rows[header_row - 1]],
    "dataframe_column": columns,
}))
display(layout.groupby("header_row").size().rename("workbooks").to_frame())

A1  Table 3. Sale prices statistics by NSW Local Government Area
A2  Reporting period: January - March 2026
A3  (s): 30 or fewer properties sold; (-): 10 or fewer properties sold
A4  Read explanatory notes
Header row in sales-tables-march-2026-quarter.xlsx is 7.


,excel_column,source_header,dataframe_column
0,A,Greater Metropolitan Region (GMR),gmr
1,B,Greater Sydney,greater_sydney
2,C,Rings,ring
3,D,Local Government Area (LGA),lga
4,E,DwellingType,dwelling_type
5,F,First Quartile Sales Price $'000s,first_quartile_price
6,G,Median Sales Price $'000s,median_price
7,H,Third Quartile Sales Price '000s,third_quartile_price
8,I,Mean Sales Price $'000s,mean_price
9,J,Sales No.,sales_count


,workbooks
header_row,
5,2
6,1
7,26


## 3. Load

Stack the published data rows from each `LGA` sheet. Use column positions because some header wording changes across issues. Keep the original cell values and attach the source filename, Excel row and reporting quarter. The `object` dtype preserves mixed Excel numbers, text and suppression markers.

In [3]:
tables = []
for filename, info in layout.iterrows():
    source_rows = lga_sheets[filename][info["header_row"]:]
    table = pd.DataFrame(source_rows, columns=columns, dtype=object)
    table.insert(0, "excel_row", range(info["header_row"] + 1, info["header_row"] + len(table) + 1))
    table.insert(0, "filename", filename)
    table.insert(0, "quarter_date", info["quarter_date"])
    # The published table has a dwelling type on each data row.
    tables.append(table.loc[table["dwelling_type"].notna()])

raw_sales = pd.concat(tables, ignore_index=True)
print("Raw LGA-sheet shape", raw_sales.shape)
display(raw_sales.dtypes.rename("dtype").to_frame())
# Show the first five rows in two parts so the full source preview fits on screen.
display(raw_sales.head().iloc[:, :10])
display(raw_sales.head().iloc[:, 10:])

Raw LGA-sheet shape (81127, 17)


,dtype
quarter_date,datetime64[us]
filename,str
excel_row,int64
gmr,object
greater_sydney,object
ring,object
lga,object
dwelling_type,object
first_quartile_price,object
median_price,object


,quarter_date,filename,excel_row,gmr,greater_sydney,ring,lga,dwelling_type,first_quartile_price,median_price
0,2019-01-01,issue-128-sales-tables-march-2019.xlsx,6,Total,Total,Total,Total,Total,455,635
1,2019-01-01,issue-128-sales-tables-march-2019.xlsx,7,Total,Total,Total,Total,Non Strata,440,630
2,2019-01-01,issue-128-sales-tables-march-2019.xlsx,8,Total,Total,Total,Total,Strata,485,640
3,2019-01-01,issue-128-sales-tables-march-2019.xlsx,9,Total,Total,Total,Albury,Total,240,320
4,2019-01-01,issue-128-sales-tables-march-2019.xlsx,10,Total,Total,Total,Albury,Non Strata,260,352


,third_quartile_price,mean_price,sales_count,quarterly_median_change,annual_median_change,quarterly_count_change,annual_count_change
0,870,764,19558,-0.0231,-0.0508,-0.2309,-0.3087
1,895,779,13598,-0.0308,-0.0526,-0.233,-0.2873
2,830,730,5960,-0.0303,-0.0448,-0.2262,-0.353
3,415,340,198,0.0492,0.0492,-0.2414,-0.2747
4,438,370,162,-0.0154,0.043,-0.243,-0.2249


## 4. Coverage

Compare the periods found in the workbook titles with the expected quarterly sequence. Count distinct labels and geography combinations. DCJ repeats LGA rows under several levels of its hierarchy. For one statewide view of the LGA rows, use the block where the first three geography columns read `Total`. For the Sydney LGA view, use the rows explicitly assigned to a ring within Greater Sydney and GMR.

In [4]:
expected = pd.period_range("2019Q1", "2026Q1", freq="Q")
observed = pd.PeriodIndex(layout["quarter"].unique()).sort_values()
print("First quarter", observed.min())
print("Latest quarter", observed.max())
print("Distinct quarters", len(observed))
print("Missing quarters", expected.difference(observed).astype(str).tolist())

geography_columns = ["gmr", "greater_sydney", "ring", "lga"]
rings = ["Inner Ring", "Middle Ring", "Outer Ring"]
gmr_name = "GMR (Greater Metropolitan Region)"
lga_rows = raw_sales.loc[raw_sales["lga"].ne("Total")]
state_lgas = lga_rows.loc[
    lga_rows["gmr"].eq("Total")
    & lga_rows["greater_sydney"].eq("Total")
    & lga_rows["ring"].eq("Total")
]
sydney_lgas = lga_rows.loc[
    lga_rows["gmr"].eq(gmr_name)
    & lga_rows["greater_sydney"].eq("Greater Sydney")
    & lga_rows["ring"].isin(rings)
]

print("Distinct A–D geography combinations", len(raw_sales[geography_columns].drop_duplicates()))
print("Distinct LGA labels", state_lgas["lga"].nunique())
print("Distinct Sydney LGA labels", sydney_lgas["lga"].nunique())
coverage = layout[["quarter", "header_row"]].copy()
coverage["lga_sheet_rows"] = raw_sales.groupby("filename").size()
coverage["distinct_lga_labels"] = state_lgas.groupby("filename")["lga"].nunique()
coverage["sydney_lga_labels"] = sydney_lgas.groupby("filename")["lga"].nunique()
display(coverage)

First quarter 2019Q1
Latest quarter 2026Q1
Distinct quarters 29
Missing quarters []


Distinct A–D geography combinations 1057
Distinct LGA labels 128
Distinct Sydney LGA labels 34


,quarter,header_row,lga_sheet_rows,distinct_lga_labels,sydney_lga_labels
filename,,,,,
issue-128-sales-tables-march-2019.xlsx,2019Q1,5,2747,127,34
issue-129-sales-tables-june-2019.xlsx,2019Q2,5,2747,127,34
issue-130-sales-tables-september-2019.xlsx,2019Q3,6,2763,127,34
issue-131-sales-tables-december-2019.xlsx,2019Q4,7,2787,127,34
issue-132-sales-tables-march-2020.xlsx,2020Q1,7,2755,126,34
issue-133-sales-tables-june-2020.xlsx,2020Q2,7,2739,127,34
issue-134-sales-tables-september-2020.xlsx,2020Q3,7,2803,128,34
issue-135-sales-tables-december-2020.xlsx,2020Q4,7,2827,127,34
issue-136-sales-tables-march-2021.xlsx,2021Q1,7,2835,128,34


## 5. Geography labels

List every LGA label and how often it appears, with DCJ’s Sydney rings shown beside the corresponding names. Compare each issue with the union of LGA labels and check whether a Sydney LGA changes rings. List the source’s aggregate geography combinations using rows where column D is `Total`.

In [5]:
sydney_membership = sydney_lgas[["lga", "ring"]].drop_duplicates()
lga_labels = (
    state_lgas.groupby("lga")["quarter_date"].nunique().rename("quarters")
    .to_frame()
    .join(sydney_membership.set_index("lga").rename(columns={"ring": "dcj_sydney_ring"}))
)
with pd.option_context("display.max_rows", None):
    display(lga_labels)

print("LGA labels absent from some quarters")
for quarter, quarter_lgas in state_lgas.groupby("quarter_date"):
    missing_labels = sorted(set(lga_labels.index) - set(quarter_lgas["lga"]))
    if missing_labels:
        print(quarter.to_period("Q"), ", ".join(missing_labels))

print("Sydney ring membership")
display(sydney_membership.groupby("ring")["lga"].nunique().rename("lgas").to_frame())
ring_changes = sydney_membership.groupby("lga")["ring"].nunique().gt(1)
print("Sydney LGAs that appear under multiple rings", ring_changes[ring_changes].index.tolist())

aggregates = raw_sales.loc[raw_sales["lga"].eq("Total"), geography_columns]
aggregate_combinations = aggregates.drop_duplicates().sort_values(geography_columns)
print("Published aggregate combinations", len(aggregate_combinations))
display(aggregate_combinations.reset_index(drop=True))

,quarters,dcj_sydney_ring
lga,,
Albury,29,NaN
Armidale Regional,29,NaN
Ballina,29,NaN
Balranald,27,NaN
Bathurst Regional,29,NaN
Bayside,29,Middle Ring
Bega Valley,29,NaN
Bellingen,29,NaN
Berrigan,29,NaN


LGA labels absent from some quarters
2019Q1 Brewarrina
2019Q2 Brewarrina
2019Q3 Brewarrina
2019Q4 Bourke
2020Q1 Brewarrina, Central Darling
2020Q2 Bogan
2020Q4 Brewarrina
2021Q2 Brewarrina
2021Q3 Brewarrina
2022Q4 Brewarrina
2023Q3 Balranald
2024Q1 Brewarrina
2024Q4 Brewarrina
2025Q2 Brewarrina
2026Q1 Balranald
Sydney ring membership


,lgas
ring,
Inner Ring,8
Middle Ring,12
Outer Ring,14


Sydney LGAs that appear under multiple rings []
Published aggregate combinations 33


,gmr,greater_sydney,ring,lga
0,GMR (Greater Metropolitan Region),Greater Sydney,Inner Ring,Total
1,GMR (Greater Metropolitan Region),Greater Sydney,Middle Ring,Total
2,GMR (Greater Metropolitan Region),Greater Sydney,Outer Ring,Total
3,GMR (Greater Metropolitan Region),Greater Sydney,Total,Total
4,GMR (Greater Metropolitan Region),Rest of GMR,Newcastle SSD,Total
5,GMR (Greater Metropolitan Region),Rest of GMR,Total,Total
6,GMR (Greater Metropolitan Region),Rest of GMR,Wollongong SSD,Total
7,GMR (Greater Metropolitan Region),Total,Inner Ring,Total
8,GMR (Greater Metropolitan Region),Total,Middle Ring,Total
9,GMR (Greater Metropolitan Region),Total,Newcastle SSD,Total


The distinct spellings are consistent across the issues checked. The LGA omissions listed above are changes in row presence. DCJ labels the non-GMR remainder `Rest of State`. Its Inner, Middle and Outer rings include Central Coast, Blue Mountains, Hawkesbury and Wollondilly in the Outer Ring.

## 6. Missing and suppressed values

Count each non-numeric marker in the four price statistics, sales count and change fields. Show the whole published sheet, the statewide LGA block and the DCJ Sydney LGA rows. Numeric strings, commas in counts and percent signs are Excel display formats. The audit recognises them without altering the raw DataFrame.

DCJ says it suppresses results for groups with 10 sales or fewer. The table uses `-` for suppression and `s` in the count field for small samples up to 30. [DCJ sales explanatory notes](https://dcj.nsw.gov.au/about-us/families-and-communities-statistics/housing-rent-and-sales/explanatory-notes.html)

In [6]:
measure_columns = columns[5:]

def as_number(values):
    text = values.astype("string").str.replace(",", "", regex=False)
    return pd.to_numeric(text.str.removesuffix("%"), errors="coerce")

def marker_counts(table):
    counts = {}
    for column in measure_columns:
        values = table[column]
        markers = values.loc[as_number(values).isna()].fillna("<blank>")
        for marker, count in markers.value_counts().items():
            counts[(column, marker)] = count
    return pd.Series(counts, dtype="int64")

markers = pd.concat({
    "all_lga_sheet_rows": marker_counts(raw_sales),
    "statewide_lga_rows": marker_counts(state_lgas),
    "dcj_sydney_lga_rows": marker_counts(sydney_lgas),
}, axis=1).reindex(measure_columns, level=0)
# A zero here is a frequency of a marker, never a replacement for a source value.
markers = markers.fillna(0).astype(int)
markers.index.names = ["measure", "marker"]
display(markers)

all_lga_sheet_rows  statewide_lga_rows  \
measure                 marker                                            
first_quartile_price    -                      8016                1002   
median_price            -                      8016                1002   
third_quartile_price    -                      8016                1002   
mean_price              -                      8016                1002   
sales_count             s                     12992                1624   
                        -                      8016                1002   
quarterly_median_change -                      8016                1002   
                        <blank>                  32                   4   
annual_median_change    -                      8016                1002   
                        <blank>                  32                   4   
quarterly_count_change  -                      8016                1002   
                        <blank>                  32                   4   
annual_count_change     -                      8016                1002   
                        <blank>                  32                   4   

                                 dcj_sydney_lga_rows  
measure                 marker                        
first_quartile_price    -                         15  
median_price            -                         15  
third_quartile_price    -                         15  
mean_price              -                         15  
sales_count             s                        133  
                        -                         15  
quarterly_median_change -                         15  
                        <blank>                    0  
annual_median_change    -                         15  
                        <blank>                    0  
quarterly_count_change  -                         15  
                        <blank>                    0  
annual_count_change     -                         15  
                        <blank>                    0

## 7. Small samples

For DCJ Sydney LGAs, summarise published numeric sales counts by dwelling type and tally the two count markers. Apply the strict fewer-than-30 threshold to numeric counts. A dash is known to be below 30. An `s` could represent exactly 30, so the workbook cannot give an exact total below 30. Check for absent LGA-quarter-type rows separately.

In [7]:
type_rows = sydney_lgas.loc[sydney_lgas["dwelling_type"].isin(["Non Strata", "Strata"])].copy()
type_rows["numeric_sales_count"] = as_number(type_rows["sales_count"])
print("Distribution of numeric sales counts per LGA and quarter")
display(type_rows.groupby("dwelling_type")["numeric_sales_count"].describe().round(1))

sample_summary = type_rows.groupby("dwelling_type").size().rename("published_rows").to_frame()
sample_summary["numeric_below_30"] = (
    type_rows["numeric_sales_count"].lt(30).groupby(type_rows["dwelling_type"]).sum()
)
for marker, column in [("-", "dash_10_or_fewer"), ("s", "small_sample_up_to_30")]:
    sample_summary[column] = (
        type_rows["sales_count"].eq(marker).groupby(type_rows["dwelling_type"]).sum()
    )
sample_summary["minimum_below_30"] = (
    sample_summary["numeric_below_30"] + sample_summary["dash_10_or_fewer"]
)
sample_summary["maximum_below_30"] = (
    sample_summary["minimum_below_30"]
    + sample_summary["small_sample_up_to_30"]
)
display(sample_summary.T)

key_columns = ["quarter_date", "lga", "dwelling_type"]
expected_keys = pd.MultiIndex.from_product(
    [
        layout["quarter_date"],
        sorted(sydney_membership["lga"].unique()),
        ["Non Strata", "Strata"],
    ],
    names=key_columns,
)
present_keys = pd.MultiIndex.from_frame(type_rows[key_columns])
duplicate_keys = present_keys.duplicated().sum()
print("Duplicate Sydney LGA-quarter-type keys", duplicate_keys)
print("Expected Sydney LGA-quarter-type keys absent from the workbooks")
missing_keys = expected_keys.difference(present_keys)
display(missing_keys.to_frame(index=False))

Distribution of numeric sales counts per LGA and quarter


,count,mean,std,min,25%,50%,75%,max
dwelling_type,,,,,,,,
Non Strata,949.0,319.1,265.5,31.0,126.0,264.0,429.0,1683.0
Strata,877.0,315.6,231.9,31.0,155.0,261.0,418.0,1354.0


dwelling_type,Non Strata,Strata
published_rows,986,985
numeric_below_30,0,0
dash_10_or_fewer,0,15
small_sample_up_to_30,37,93
minimum_below_30,0,15
maximum_below_30,37,108


Duplicate Sydney LGA-quarter-type keys 0
Expected Sydney LGA-quarter-type keys absent from the workbooks


,quarter_date,lga,dwelling_type
0,2019-04-01,Wollondilly,Strata


## 8. Spot-check candidates

Locate two median prices and one sales count in the latest workbook. Use the Greater Sydney ring rows. Median values are in thousands of dollars and the count is in sales.

In [8]:
checks = [
    ("Blacktown", "Non Strata", "median_price"),
    ("Sydney", "Strata", "median_price"),
    ("Parramatta", "Strata", "sales_count"),
]
latest_sydney = sydney_lgas.loc[sydney_lgas["filename"].eq(representative)]
spot_checks = []
for lga, dwelling_type, measure in checks:
    matching_rows = (
        latest_sydney["lga"].eq(lga)
        & latest_sydney["dwelling_type"].eq(dwelling_type)
    )
    row = latest_sydney.loc[matching_rows].iloc[0]
    column_letter = excel_columns[columns.index(measure)]
    spot_checks.append({
        "lga": lga,
        "quarter_date": row["quarter_date"],
        "dwelling_type": dwelling_type,
        "measure": measure,
        "raw_value": row[measure],
        "cell": f"{column_letter}{row['excel_row']}",
    })
print(f"Workbook {representative}\nWorksheet LGA")
display(pd.DataFrame(spot_checks).set_index("lga"))

Workbook sales-tables-march-2026-quarter.xlsx
Worksheet LGA


,quarter_date,dwelling_type,measure,raw_value,cell
lga,,,,,
Blacktown,2026-01-01,Non Strata,median_price,1250,G1865
Sydney,2026-01-01,Strata,median_price,967,G1815
Parramatta,2026-01-01,Strata,sales_count,835,J1851
